# Train LoRA - Old Photo Repair Inpainting

This notebook fine-tunes a LoRA adapter for `runwayml/stable-diffusion-inpainting` on the filtered OpenPhoto inpainting dataset.

Training task: `damaged image + derived mask + prompt -> pristine image`.

Before running, add the processed Kaggle Dataset as an input:

https://www.kaggle.com/datasets/dwctien/openphoto-restore-filtered-inpainting-subset

After it is added to the session, Kaggle usually mounts it at:

```text
/kaggle/input/openphoto-restore-filtered-inpainting-subset
```

## 0. Setup

Recommended Kaggle settings:

- Accelerator: GPU, preferably T4/P100 or better.
- Internet: ON for downloading the pretrained model and Python packages.
- Add the processed Kaggle dataset `openphoto-restore-filtered-inpainting-subset` as session input: https://www.kaggle.com/datasets/dwctien/openphoto-restore-filtered-inpainting-subset

The notebook saves LoRA weights, validation grids, a small baseline-vs-LoRA evaluation table, and a zip archive under `/kaggle/working/outputs_old_photo_lora_v2`.

In [ ]:
!pip install -q diffusers transformers accelerate peft safetensors pandas pillow tqdm matplotlib torchvision scikit-image

In [ ]:
from pathlib import Path
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

from diffusers import StableDiffusionInpaintPipeline, DDPMScheduler
from diffusers.optimization import get_scheduler
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Configuration

This configuration focuses on local old-photo damage by using stricter mask filtering, a low learning rate, a small LoRA rank, and periodic checkpoints.

In [ ]:
DATASET_SLUG = 'openphoto-restore-filtered-inpainting-subset'
DATASET_CANDIDATES = [
    Path('/kaggle/input') / DATASET_SLUG,
]

DATASET_ROOT = None
for candidate in DATASET_CANDIDATES:
    if (candidate / 'metadata_all.csv').exists():
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    matches = list(Path('/kaggle/input').glob('**/metadata_all.csv'))
    assert matches, 'Dataset not found. Add openphoto-restore-filtered-inpainting-subset as Kaggle input first.'
    DATASET_ROOT = matches[0].parent

TRAIN_ROOT = DATASET_ROOT / 'train'
TEST_ROOT = DATASET_ROOT / 'test'
TRAIN_METADATA_PATH = TRAIN_ROOT / 'metadata.csv'
TEST_METADATA_PATH = TEST_ROOT / 'metadata.csv'

OUTPUT_ROOT = Path('/kaggle/working/outputs_old_photo_lora_v2')
LORA_OUT_DIR = OUTPUT_ROOT / 'lora_final'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
SAMPLE_DIR = OUTPUT_ROOT / 'samples'
PREFLIGHT_DIR = OUTPUT_ROOT / 'preflight'
LOG_DIR = OUTPUT_ROOT / 'logs'
EVAL_DIR = OUTPUT_ROOT / 'quick_eval'

for d in [LORA_OUT_DIR, CHECKPOINT_DIR, SAMPLE_DIR, PREFLIGHT_DIR, LOG_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'runwayml/stable-diffusion-inpainting'
PROMPT = 'a realistic restored old photo'
NEGATIVE_PROMPT = 'blurry, distorted, low quality, artifacts, color spots, noisy texture'

IMAGE_SIZE = 512
SEED = 42

LORA_RANK = 4
LORA_ALPHA = 4
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 800
LR_WARMUP_STEPS = 80
NUM_WORKERS = 2

CHECKPOINT_EVERY_STEPS = 200
VALIDATION_EVERY_STEPS = 200
NUM_VALIDATION_IMAGES = 4
INFERENCE_STEPS_FOR_VALIDATION = 25
GUIDANCE_SCALE = 7.5

QUICK_EVAL_SAMPLES = 60
QUICK_EVAL_INFERENCE_STEPS = 25

device = 'cuda' if torch.cuda.is_available() else 'cpu'
weight_dtype = torch.float16 if device == 'cuda' else torch.float32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)
print('Device:', device, 'dtype:', weight_dtype)

## 2. Load Dataset Metadata

In [ ]:
assert TRAIN_METADATA_PATH.exists(), f'Missing train metadata: {TRAIN_METADATA_PATH}'
assert TEST_METADATA_PATH.exists(), f'Missing test metadata: {TEST_METADATA_PATH}'

train_df = pd.read_csv(TRAIN_METADATA_PATH)
test_df = pd.read_csv(TEST_METADATA_PATH)

def add_abs_paths(frame, split_root):
    frame = frame.copy()
    frame['damaged_abs'] = frame['damaged_path'].apply(lambda p: str(DATASET_ROOT / p))
    frame['pristine_abs'] = frame['pristine_path'].apply(lambda p: str(DATASET_ROOT / p))
    frame['mask_abs'] = frame['mask_path'].apply(lambda p: str(DATASET_ROOT / p))
    return frame

train_df = add_abs_paths(train_df, TRAIN_ROOT)
test_df = add_abs_paths(test_df, TEST_ROOT)

# Use stricter masks than the exported dataset to reduce color speckles and global restoration leakage.
TRAIN_MIN_MASK_AREA = 0.005
TRAIN_MAX_MASK_AREA = 0.25
TEST_MIN_MASK_AREA = 0.005
TEST_MAX_MASK_AREA = 0.25

original_train_count = len(train_df)
original_test_count = len(test_df)
train_df = train_df[
    train_df['mask_area'].between(TRAIN_MIN_MASK_AREA, TRAIN_MAX_MASK_AREA)
].copy().reset_index(drop=True)
test_df = test_df[
    test_df['mask_area'].between(TEST_MIN_MASK_AREA, TEST_MAX_MASK_AREA)
].copy().reset_index(drop=True)

assert len(train_df) > 0, 'No train samples left after mask filtering.'
assert len(test_df) > 0, 'No test samples left after mask filtering.'

for frame_name, frame in [('train', train_df), ('test', test_df)]:
    for col in ['damaged_abs', 'pristine_abs', 'mask_abs']:
        missing = [p for p in frame[col].head(20) if not Path(p).exists()]
        assert not missing, f'Missing files in {frame_name}/{col}: {missing[:3]}'

print('Train samples:', len(train_df), f'(filtered from {original_train_count})')
print('Test samples:', len(test_df), f'(filtered from {original_test_count})')
print('Train mask area:')
display(train_df['mask_area'].describe())
print('Test mask area:')
display(test_df['mask_area'].describe())

## 3. Dataset And Preflight Preview

Panel order: `pristine | damaged | mask | masked damaged`.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

class OldPhotoInpaintDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        pristine = Image.open(row['pristine_abs']).convert('RGB')
        damaged = Image.open(row['damaged_abs']).convert('RGB')
        mask = Image.open(row['mask_abs']).convert('L')

        pristine_tensor = image_transform(pristine)
        damaged_tensor = image_transform(damaged)
        mask_tensor = (mask_transform(mask) > 0.5).float()
        masked_damaged_tensor = damaged_tensor * (1.0 - mask_tensor)

        return {
            'pristine': pristine_tensor,
            'damaged': damaged_tensor,
            'mask': mask_tensor,
            'masked_damaged': masked_damaged_tensor,
            'id': int(row['id']),
        }

train_dataset = OldPhotoInpaintDataset(train_df)
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print('Batches per epoch:', len(train_loader))

In [ ]:
def tensor_to_pil(t):
    t = t.detach().cpu().float()
    if t.shape[0] == 1:
        arr = (t.squeeze(0).numpy() * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(arr, mode='L')
    t = (t + 1.0) / 2.0
    arr = (t.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr, mode='RGB')

batch = next(iter(train_loader))
mask_ratio = float(batch['mask'][0].mean().item())
assert 0.001 <= mask_ratio <= 0.7, f'Unexpected mask ratio: {mask_ratio}'

panels = [
    tensor_to_pil(batch['pristine'][0]),
    tensor_to_pil(batch['damaged'][0]),
    tensor_to_pil(batch['mask'][0]).convert('RGB'),
    tensor_to_pil(batch['masked_damaged'][0]),
]
grid = Image.new('RGB', (IMAGE_SIZE * len(panels), IMAGE_SIZE), 'white')
for i, panel in enumerate(panels):
    grid.paste(panel.convert('RGB'), (i * IMAGE_SIZE, 0))

preflight_path = PREFLIGHT_DIR / 'train_batch_preview.png'
grid.save(preflight_path)

plt.figure(figsize=(16, 4))
plt.imshow(grid)
plt.axis('off')
plt.show()
print('Saved:', preflight_path)
print('Mask ratio:', round(mask_ratio, 4))

## 4. Load Model And Attach LoRA

In [ ]:
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=weight_dtype,
    safety_checker=None,
)

vae = pipe.vae.to(device, dtype=weight_dtype)
text_encoder = pipe.text_encoder.to(device, dtype=weight_dtype)
tokenizer = pipe.tokenizer
unet = pipe.unet.to(device, dtype=weight_dtype)
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    init_lora_weights='gaussian',
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'],
)
unet.add_adapter(lora_config)

for p in unet.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainable_params = [p for p in unet.parameters() if p.requires_grad]
print('Trainable LoRA parameters:', sum(p.numel() for p in trainable_params))

if hasattr(unet, 'enable_gradient_checkpointing'):
    unet.enable_gradient_checkpointing()
if hasattr(pipe, 'enable_attention_slicing'):
    pipe.enable_attention_slicing()
if hasattr(pipe, 'enable_vae_slicing'):
    pipe.enable_vae_slicing()

## 5. Training Utilities

In [ ]:
def encode_prompt(prompt, batch_size):
    text_inputs = tokenizer(
        [prompt] * batch_size,
        padding='max_length',
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors='pt',
    )
    text_input_ids = text_inputs.input_ids.to(device)
    with torch.no_grad():
        prompt_embeds = text_encoder(text_input_ids)[0]
    return prompt_embeds

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    eps=1e-8,
)

lr_scheduler = get_scheduler(
    'constant_with_warmup',
    optimizer=optimizer,
    num_warmup_steps=LR_WARMUP_STEPS,
    num_training_steps=MAX_TRAIN_STEPS,
)

def save_lora(save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    unet_lora_state_dict = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionInpaintPipeline.save_lora_weights(
        save_directory=str(save_dir),
        unet_lora_layers=unet_lora_state_dict,
        safe_serialization=True,
    )

def load_rgb(path):
    return Image.open(path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

def load_mask(path):
    mask = Image.open(path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
    arr = (np.asarray(mask) > 127).astype(np.uint8) * 255
    return Image.fromarray(arr, mode='L')

def make_grid(panels):
    grid = Image.new('RGB', (IMAGE_SIZE * len(panels), IMAGE_SIZE), 'white')
    for i, panel in enumerate(panels):
        grid.paste(panel.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE)), (i * IMAGE_SIZE, 0))
    return grid

validation_rows = test_df.sample(min(NUM_VALIDATION_IMAGES, len(test_df)), random_state=SEED).reset_index(drop=True)

@torch.no_grad()
def run_validation(step):
    save_lora(CHECKPOINT_DIR / f'step_{step}')

    val_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=weight_dtype,
        safety_checker=None,
    ).to(device)
    val_pipe.load_lora_weights(str(CHECKPOINT_DIR / f'step_{step}'))
    val_pipe.enable_attention_slicing()
    if hasattr(val_pipe, 'enable_vae_slicing'):
        val_pipe.enable_vae_slicing()

    step_dir = SAMPLE_DIR / f'step_{step}'
    step_dir.mkdir(parents=True, exist_ok=True)

    for _, row in validation_rows.iterrows():
        pristine = load_rgb(row['pristine_abs'])
        damaged = load_rgb(row['damaged_abs'])
        mask = load_mask(row['mask_abs'])
        generator = torch.Generator(device=device).manual_seed(SEED + int(row['id']))
        pred = val_pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=damaged,
            mask_image=mask,
            num_inference_steps=INFERENCE_STEPS_FOR_VALIDATION,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]
        grid = make_grid([pristine, damaged, mask.convert('RGB'), pred])
        grid.save(step_dir / f"sample_{int(row['id']):06d}_grid.png")

    del val_pipe
    torch.cuda.empty_cache()
    print('Saved validation samples:', step_dir)

## 6. Train LoRA

In [ ]:
unet.train()
vae.eval()
text_encoder.eval()

global_step = 0
micro_step = 0
running_loss = 0.0
log_rows = []
start_time = time.time()

progress_bar = tqdm(total=MAX_TRAIN_STEPS, desc='Training LoRA')
optimizer.zero_grad(set_to_none=True)

while global_step < MAX_TRAIN_STEPS:
    for batch in train_loader:
        pristine = batch['pristine'].to(device=device, dtype=weight_dtype)
        masked_damaged = batch['masked_damaged'].to(device=device, dtype=weight_dtype)
        mask = batch['mask'].to(device=device, dtype=weight_dtype)
        batch_size = pristine.shape[0]

        with torch.no_grad():
            latents = vae.encode(pristine).latent_dist.sample() * vae.config.scaling_factor
            masked_image_latents = vae.encode(masked_damaged).latent_dist.sample() * vae.config.scaling_factor

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (batch_size,),
            device=device,
            dtype=torch.long,
        )
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        mask_latents = F.interpolate(mask, size=latents.shape[-2:], mode='nearest')
        model_input = torch.cat([noisy_latents, mask_latents, masked_image_latents], dim=1)
        prompt_embeds = encode_prompt(PROMPT, batch_size).to(dtype=weight_dtype)

        model_pred = unet(model_input, timesteps, encoder_hidden_states=prompt_embeds).sample
        loss = F.mse_loss(model_pred.float(), noise.float(), reduction='mean')
        if not torch.isfinite(loss):
            save_lora(CHECKPOINT_DIR / f'failed_step_{global_step}_nan_guard')
            raise RuntimeError(f'Non-finite loss at step {global_step}: {loss.item()}')

        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        running_loss += loss.detach().item() * GRADIENT_ACCUMULATION_STEPS
        micro_step += 1

        if micro_step % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            mean_loss = running_loss / max(micro_step, 1)
            progress_bar.update(1)
            progress_bar.set_postfix(loss=mean_loss, lr=lr_scheduler.get_last_lr()[0])

            if global_step % 50 == 0:
                log_rows.append({
                    'step': global_step,
                    'loss_running_mean': mean_loss,
                    'lr': lr_scheduler.get_last_lr()[0],
                    'elapsed_sec': time.time() - start_time,
                })
                pd.DataFrame(log_rows).to_csv(LOG_DIR / 'training_log.csv', index=False)

            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_lora(CHECKPOINT_DIR / f'step_{global_step}')
                print('Saved checkpoint:', CHECKPOINT_DIR / f'step_{global_step}')

            if global_step % VALIDATION_EVERY_STEPS == 0:
                unet.eval()
                run_validation(global_step)
                unet.train()

        if global_step >= MAX_TRAIN_STEPS:
            break

progress_bar.close()
save_lora(LORA_OUT_DIR)
print('Saved final LoRA:', LORA_OUT_DIR)

## 7. Training Curve

In [ ]:
log_path = LOG_DIR / 'training_log.csv'
if log_path.exists():
    log_df = pd.read_csv(log_path)
    display(log_df.tail())
    plt.figure(figsize=(7, 4))
    plt.plot(log_df['step'], log_df['loss_running_mean'])
    plt.xlabel('Step')
    plt.ylabel('Running mean loss')
    plt.title('Old photo LoRA training loss')
    plt.show()
else:
    print('No training log found.')

## 8. Quick Test Evaluation

This is intentionally small so it can finish on Kaggle. It compares damaged input, pretrained inpainting, and LoRA inpainting on the held-out `test` split.

For fair image-level metrics, outputs are hard-composited: generated pixels inside the mask, original damaged pixels outside the mask.

In [ ]:
def hard_composite(generated, damaged, mask):
    gen = np.asarray(generated.convert('RGB')).astype(np.uint8)
    dam = np.asarray(damaged.convert('RGB')).astype(np.uint8)
    m = np.asarray(mask.convert('L')) > 127
    out = dam.copy()
    out[m] = gen[m]
    return Image.fromarray(out, mode='RGB')

def compute_metrics(pred, target):
    pred_arr = np.asarray(pred.convert('RGB')).astype(np.float32) / 255.0
    tgt_arr = np.asarray(target.convert('RGB')).astype(np.float32) / 255.0
    return {
        'psnr': peak_signal_noise_ratio(tgt_arr, pred_arr, data_range=1.0),
        'ssim': structural_similarity(tgt_arr, pred_arr, channel_axis=2, data_range=1.0),
    }

@torch.no_grad()
def run_quick_eval():
    eval_rows = test_df.sample(min(QUICK_EVAL_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

    pretrained_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=weight_dtype,
        safety_checker=None,
    ).to(device)
    pretrained_pipe.enable_attention_slicing()
    if hasattr(pretrained_pipe, 'enable_vae_slicing'):
        pretrained_pipe.enable_vae_slicing()

    lora_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=weight_dtype,
        safety_checker=None,
    ).to(device)
    lora_pipe.load_lora_weights(str(LORA_OUT_DIR))
    lora_pipe.enable_attention_slicing()
    if hasattr(lora_pipe, 'enable_vae_slicing'):
        lora_pipe.enable_vae_slicing()

    rows = []
    for i, row in tqdm(eval_rows.iterrows(), total=len(eval_rows), desc='Quick eval'):
        pristine = load_rgb(row['pristine_abs'])
        damaged = load_rgb(row['damaged_abs'])
        mask = load_mask(row['mask_abs'])

        generator_pre = torch.Generator(device=device).manual_seed(SEED + i)
        generator_lora = torch.Generator(device=device).manual_seed(SEED + i)

        pretrained_raw = pretrained_pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=damaged,
            mask_image=mask,
            num_inference_steps=QUICK_EVAL_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator_pre,
        ).images[0]
        lora_raw = lora_pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=damaged,
            mask_image=mask,
            num_inference_steps=QUICK_EVAL_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator_lora,
        ).images[0]

        pretrained_comp = hard_composite(pretrained_raw, damaged, mask)
        lora_comp = hard_composite(lora_raw, damaged, mask)

        damaged_metrics = compute_metrics(damaged, pristine)
        pretrained_metrics = compute_metrics(pretrained_comp, pristine)
        lora_metrics = compute_metrics(lora_comp, pristine)

        rows.extend([
            {'sample_id': int(row['id']), 'method': 'damaged', **damaged_metrics},
            {'sample_id': int(row['id']), 'method': 'pretrained_hard_composite', **pretrained_metrics},
            {'sample_id': int(row['id']), 'method': 'lora_hard_composite', **lora_metrics},
        ])

        if i < 8:
            grid = make_grid([pristine, damaged, mask.convert('RGB'), pretrained_comp, lora_comp])
            grid.save(EVAL_DIR / f'eval_sample_{int(row["id"]):06d}_grid.png')

    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv(EVAL_DIR / 'quick_eval_metrics.csv', index=False)
    summary = metrics_df.groupby('method')[['psnr', 'ssim']].mean().sort_index()
    summary.to_csv(EVAL_DIR / 'quick_eval_summary.csv')

    del pretrained_pipe, lora_pipe
    torch.cuda.empty_cache()
    return metrics_df, summary

metrics_df, summary = run_quick_eval()
display(summary)
print('Saved quick eval outputs:', EVAL_DIR)

## 9. Package Outputs

Save this zip as a Kaggle notebook output or publish it as a dataset if you want to evaluate in a separate notebook.

In [ ]:
archive_base = Path('/kaggle/working/old_photo_lora_v2_outputs')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT)
print('Final LoRA:', LORA_OUT_DIR)
print('Training log:', LOG_DIR / 'training_log.csv')
print('Quick eval summary:', EVAL_DIR / 'quick_eval_summary.csv')
print('Output archive:', archive_path)